In [1]:
!pip install imagehash
import tensorflow as tf
import numpy as np
import cv2
import os
import zipfile
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from collections import defaultdict, Counter
import imagehash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 22.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from google.colab import drive
drive.flush_and_unmount()

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
zip_path = "/content/drive/MyDrive/Parkinson's Dataset Final.zip"
extract_path = "/content/dataset"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

BASE = "/content/dataset/Parkinson's Dataset Final"
print(os.listdir(BASE))

['Hand Pd', 'New hand pd Dataset']


In [7]:
HANDPD_ROOT = os.path.join(BASE, "Hand Pd")

SPIRAL_ROOT = os.path.join(HANDPD_ROOT, "Spiral_HandPD")
MEANDER_ROOT = os.path.join(HANDPD_ROOT, "Meander_HandPD")

image_paths = []
labels = []

def collect_handpd(root):

    for cls in os.listdir(root):

        cls_path = os.path.join(root, cls)

        if not os.path.isdir(cls_path):
            continue

        if cls.lower() == "healthy":
            lab = 0
        elif cls.lower() == "parkinson":
            lab = 1
        else:
            continue

        for r, d, files in os.walk(cls_path):

            for f in files:

                if f.lower().endswith((".png",".jpg",".jpeg",".bmp")):

                    image_paths.append(os.path.join(r,f))
                    labels.append(lab)

collect_handpd(SPIRAL_ROOT)
collect_handpd(MEANDER_ROOT)

print("Total Images:",len(image_paths))
print("Healthy:",labels.count(0))
print("Parkinson:",labels.count(1))

Total Images: 808
Healthy: 216
Parkinson: 592


In [8]:
def get_phash(img_path):

    try:
        img = Image.open(img_path).convert("L").resize((224,224))
        return imagehash.phash(img)
    except:
        return None

hash_dict = defaultdict(list)

for p in tqdm(image_paths):

    h = get_phash(p)

    if h:
        hash_dict[str(h)].append(p)

clean_paths=[]
clean_labels=[]

path_label = dict(zip(image_paths,labels))

for h in hash_dict:

    p = hash_dict[h][0]

    clean_paths.append(p)
    clean_labels.append(path_label[p])

print("After Cleaning:",len(clean_paths))

100%|██████████| 808/808 [00:08<00:00, 89.90it/s] 

After Cleaning: 735


In [9]:
paths=np.array(clean_paths)
labels=np.array(clean_labels)

train_paths,test_paths,y_train,y_test=train_test_split(
paths,labels,
test_size=0.2,
stratify=labels,
random_state=42
)

val_paths,test_paths,y_val,y_test=train_test_split(
test_paths,y_test,
test_size=0.5,
stratify=y_test,
random_state=42
)

print("Train:",Counter(y_train))
print("Val:",Counter(y_val))
print("Test:",Counter(y_test))

Train: Counter({np.int64(1): 473, np.int64(0): 115})
Val: Counter({np.int64(1): 59, np.int64(0): 14})
Test: Counter({np.int64(1): 59, np.int64(0): 15})


In [15]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

{0: np.float64(2.5565217391304347), 1: np.float64(0.6215644820295984)}


In [10]:
IMG_SIZE=224
BATCH_SIZE=16

def preprocess(path):

    path=path.numpy().decode()

    img=cv2.imread(path,cv2.IMREAD_GRAYSCALE)
    img=cv2.resize(img,(IMG_SIZE,IMG_SIZE))

    clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
    img=clahe.apply(img)

    img=img/255.0
    img=np.expand_dims(img,-1)

    return img.astype(np.float32)

def tf_preprocess(path,label):

    img=tf.py_function(preprocess,[path],tf.float32)
    img.set_shape((IMG_SIZE,IMG_SIZE,1))

    return img,label

In [11]:
def make_ds(paths,labels):

    ds=tf.data.Dataset.from_tensor_slices((paths,labels))

    ds=ds.map(tf_preprocess,num_parallel_calls=tf.data.AUTOTUNE)

    ds=ds.batch(BATCH_SIZE)

    ds=ds.prefetch(tf.data.AUTOTUNE)

    return ds

train_ds=make_ds(train_paths,y_train)
val_ds=make_ds(val_paths,y_val)
test_ds=make_ds(test_paths,y_test)

In [18]:
from tensorflow.keras import layers
from tensorflow import keras
import tensorflow as tf

IMG_SIZE = 224
PATCH_SIZE = 16
EMBED_DIM = 128


def build_hybrid_model():

    inputs = layers.Input((IMG_SIZE, IMG_SIZE, 1))

    # CNN Feature Extractor
    x = layers.Conv2D(32,3,activation='relu',padding='same')(inputs)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64,3,activation='relu',padding='same')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128,3,activation='relu',padding='same')(x)

    # Patch embedding
    x = layers.Conv2D(
        filters=EMBED_DIM,
        kernel_size=PATCH_SIZE,
        strides=PATCH_SIZE
    )(x)

    shape = x.shape

    x = layers.Reshape((-1, EMBED_DIM))(x)

    # Transformer
    attn = layers.MultiHeadAttention(num_heads=4,key_dim=32)(x,x)

    x = layers.Add()([x,attn])
    x = layers.LayerNormalization()(x)

    ffn = layers.Dense(256,activation='gelu')(x)
    ffn = layers.Dense(EMBED_DIM)(ffn)

    x = layers.Add()([x,ffn])
    x = layers.LayerNormalization()(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(128,activation='relu')(x)

    outputs = layers.Dense(2,activation='softmax')(x)

    model = keras.Model(inputs,outputs)

    return model


model = build_hybrid_model()

model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 224, 224,  │        320 │ input_layer_5[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 112, 112,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 56, 56,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 3, 3, 128) │  4,194,432 │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 9, 128)    │          0 │ conv2d_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 9, 128)    │     66,048 │ reshape[0][0],    │
│ (MultiHeadAttentio… │                   │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 9, 128)    │          0 │ reshape[0][0],    │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 9, 128)    │        256 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 9, 256)    │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 9, 128)    │     32,896 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 9, 128)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_11[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 9, 128)    │        256 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 128)       │     16,512 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 2)         │        258 │ dense_12[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,436,354 (16.92 MB)

 Trainable params: 4,436,354 (16.92 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
model.compile(
optimizer=tf.keras.optimizers.Adam(1e-4),
loss="sparse_categorical_crossentropy",
metrics=["accuracy"]
)

In [24]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=6,
    restore_best_weights=True
)

In [25]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=[early_stop]
)

Epoch 1/40
37/37 ━━━━━━━━━━━━━━━━━━━━ 6s 172ms/step - accuracy: 1.0000 - loss: 4.1620e-04 - val_accuracy: 0.9041 - val_loss: 0.5019
Epoch 2/40
37/37 ━━━━━━━━━━━━━━━━━━━━ 8s 204ms/step - accuracy: 1.0000 - loss: 3.5773e-04 - val_accuracy: 0.9041 - val_loss: 0.5077
Epoch 3/40
37/37 ━━━━━━━━━━━━━━━━━━━━ 6s 169ms/step - accuracy: 1.0000 - loss: 3.1296e-04 - val_accuracy: 0.9041 - val_loss: 0.5133
Epoch 4/40
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 199ms/step - accuracy: 1.0000 - loss: 2.7745e-04 - val_accuracy: 0.9041 - val_loss: 0.5179
Epoch 5/40
37/37 ━━━━━━━━━━━━━━━━━━━━ 9s 174ms/step - accuracy: 1.0000 - loss: 2.4836e-04 - val_accuracy: 0.9041 - val_loss: 0.5225
Epoch 6/40
37/37 ━━━━━━━━━━━━━━━━━━━━ 7s 196ms/step - accuracy: 1.0000 - loss: 2.2452e-04 - val_accuracy: 0.9041 - val_loss: 0.5268
Epoch 7/40
37/37 ━━━━━━━━━━━━━━━━━━━━ 10s 197ms/step - accuracy: 1.0000 - loss: 2.0385e-04 - val_accuracy: 0.9041 - val_loss: 0.5303


In [26]:
model.evaluate(test_ds)

5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 849ms/step - accuracy: 0.8213 - loss: 0.9936


[0.7620887160301208, 0.8648648858070374]

In [27]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_true = np.concatenate([y for x, y in test_ds], axis=0)

pred = model.predict(test_ds)
y_pred = np.argmax(pred, axis=1)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 806ms/step
[[ 8  7]
 [ 3 56]]
              precision    recall  f1-score   support

           0       0.73      0.53      0.62        15
           1       0.89      0.95      0.92        59

    accuracy                           0.86        74
   macro avg       0.81      0.74      0.77        74
weighted avg       0.86      0.86      0.86        74

